<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part C: Machine Learning Approaches</h2>
<h2>Notebook C02: Machine Learning Models for Forecasting</h2>
</div>

Notebook C01 built the feature matrix. This one puts models on it: linear regression with and without
regularisation, decision trees, random forests, and the gradient boosting libraries.

The interesting question is not which of them wins among themselves. It is how they compare with the
statistical models of Part B and with the baselines that have repeatedly embarrassed both. For the first
time in this course, the sophisticated approach wins clearly, and the second half of the notebook is
about why, and about the one thing these models cannot do at all.

---

**Contents**

1. [Imports and the Feature Matrix](#1.-Imports-and-the-Feature-Matrix)
2. [Linear Models and Regularisation](#2.-Linear-Models-and-Regularisation)
3. [Trees and Random Forests](#3.-Trees-and-Random-Forests)
4. [Gradient Boosted Trees](#4.-Gradient-Boosted-Trees)
5. [Tuning, and What a Validation Score Is Worth](#5.-Tuning,-and-What-a-Validation-Score-Is-Worth)
6. [The Full Comparison](#6.-The-Full-Comparison)
7. [Where Trees Fail: Extrapolation](#7.-Where-Trees-Fail:-Extrapolation)
8. [Choosing a Model](#8.-Choosing-a-Model)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-the-Feature-Matrix">1. Imports and the Feature Matrix</h3>
</div>

In [ ]:
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from statsmodels.tsa.arima.model import ARIMA

import nb_config

sns.set_theme(style="whitegrid")

The same store and the same features as Notebook [C01](./C01_Feature_engineering.ipynb), rebuilt here so
this notebook stands alone. Every rolling statistic is shifted before it is rolled, for the reasons that
notebook spent a section on.

In [ ]:
sales = pd.read_csv(nb_config.ROSSMANN_TRAIN_PATH, parse_dates=["Date"], low_memory=False)
store = sales[sales["Store"] == 1].set_index("Date").sort_index().asfreq("D")
target = store["Sales"].astype(float)


def build_features(target, store):
    """Lags, shifted rolling statistics, calendar terms and known-in-advance columns."""
    features = pd.DataFrame(index=target.index)

    for lag in (1, 2, 7, 14, 28):
        features[f"lag_{lag}"] = target.shift(lag)

    history = target.shift(1)
    for window in (7, 28):
        features[f"roll_mean_{window}"] = history.rolling(window).mean()
        features[f"roll_std_{window}"] = history.rolling(window).std()

    features["day_of_week"] = target.index.dayofweek
    features["day_of_month"] = target.index.day
    features["month"] = target.index.month
    features["days_since_start"] = (target.index - target.index[0]).days

    for k in (1, 2):
        position = target.index.dayofyear / 365.25
        features[f"fourier_sin_{k}"] = np.sin(2 * np.pi * k * position)
        features[f"fourier_cos_{k}"] = np.cos(2 * np.pi * k * position)

    features["open"] = store["Open"]
    features["promo"] = store["Promo"]
    features["school_holiday"] = store["SchoolHoliday"]

    return features


features = build_features(target, store)
complete = features.notna().all(axis=1)
X, y = features[complete], target[complete]

HOLDOUT_DAYS = 90
split = len(X) - HOLDOUT_DAYS

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

print(f"{X.shape[1]} features, {len(X_train)} training rows, {len(X_test)} test days")
print(f"Test period: {X_test.index.min().date()} to {X_test.index.max().date()}")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-Linear-Models-and-Regularisation">2. Linear Models and Regularisation</h3>
</div>

Start with the simplest thing that could work. **Linear regression** fits a weighted sum of the features,
which for lag features is very nearly an autoregressive model fitted by least squares.

Two refinements matter in practice. **Scaling** comes first: lag features run in the thousands while the
Fourier terms sit between -1 and 1, and any penalty on coefficient size would otherwise fall almost
entirely on the Fourier terms. Then **regularisation**, which penalises large coefficients to stop the
model chasing noise:

- **Ridge** (L2) shrinks coefficients smoothly toward zero without ever reaching it.
- **Lasso** (L1) can push coefficients to exactly zero, which makes it a feature selector as well.

`make_pipeline` keeps the scaler attached to the model, so the scaling is refitted on each training fold
rather than computed once over all the data. That is the same leakage discipline as C01, applied to
preprocessing.

In [ ]:
def evaluate(model, X_train, y_train, X_test, y_test):
    """Fit and score on the held-out period."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit(X_train, y_train)
    return mean_absolute_error(y_test, model.predict(X_test))


linear_models = {
    "Linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10.0)),
    "Lasso": make_pipeline(StandardScaler(), Lasso(alpha=10.0, max_iter=5000)),
}

for name, model in linear_models.items():
    print(f"{name:<20} MAE = {evaluate(model, X_train, y_train, X_test, y_test):6.1f}")

In [ ]:
ridge = linear_models["Ridge"]
coefficients = pd.Series(
    ridge.named_steps["ridge"].coef_, index=X.columns
).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))

top = coefficients.head(12).iloc[::-1]
colours = ["seagreen" if value > 0 else "crimson" for value in top]
ax.barh(top.index, top.values, color=colours)
ax.axvline(0, color="black", linewidth=1.0)
ax.set_title("Ridge coefficients on standardised features", fontsize=13, fontweight="bold")
ax.set_xlabel("Coefficient")
ax.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The three linear models land within 12 MAE of each other, so regularisation is barely doing anything
here. That is what you would expect with 20 features and 824 rows: there is not enough freedom to overfit
badly in the first place. Regularisation earns its place when features outnumber observations, or when
they are strongly collinear, and it costs nothing to include.

The coefficients are readable, which is the main reason to keep a linear model around. `open` dominates
by a wide margin and in the expected direction, and `promo` follows. This is the same story the
permutation importances told in C01 and the SARIMAX coefficients told in B02, now as a third independent
confirmation.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-Trees-and-Random-Forests">3. Trees and Random Forests</h3>
</div>

A **decision tree** splits the feature space on thresholds: *if `open` is 0, predict near zero; otherwise
if `promo` is 1 and it is a Monday, predict higher*. That structure captures two things a linear model
cannot: **non-linearity**, and **interactions** between features, without your having to name either in
advance.

The weakness is variance. A tree grown to full depth memorises the training set, and a small change to the
data produces a completely different tree.

A **random forest** fixes that by averaging many trees, each grown on a bootstrap sample of the rows and
allowed to consider only a random subset of features at each split. The trees are individually worse and
collectively far better, because their errors are partly independent and averaging cancels them.

In [ ]:
tree_models = {
    "Decision tree (full depth)": DecisionTreeRegressor(random_state=0),
    "Decision tree (depth 5)": DecisionTreeRegressor(max_depth=5, random_state=0),
    "Random forest": RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
}

for name, model in tree_models.items():
    score = evaluate(model, X_train, y_train, X_test, y_test)
    training_score = mean_absolute_error(y_train, model.predict(X_train))
    print(f"{name:<28} train MAE = {training_score:6.1f}   test MAE = {score:6.1f}")

The full-depth tree scores a training MAE of essentially zero and 366 on the test set. That gap is
overfitting in its purest form: the tree has memorised every training day and learned nothing that
generalises. Limiting the depth closes most of the gap, at the cost of some flexibility.

The random forest is better than either, and notice that its training error is *also* low without the
same penalty on the test set. Averaging across 300 differently-grown trees is what buys that: no single
tree's memorisation survives the average.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Gradient-Boosted-Trees">4. Gradient Boosted Trees</h3>
</div>

A random forest grows its trees **independently** and averages them. **Gradient boosting** grows them
**sequentially**, each new tree fitted to the errors the previous ones left behind. The trees are
deliberately shallow, and the model improves by accumulating many small corrections.

That usually buys accuracy, and it costs robustness: a boosted model with too many trees or too high a
learning rate will fit the noise, where a forest simply cannot. Boosting needs tuning; forests mostly do
not.

Three implementations matter in practice, and they agree far more than the marketing suggests:

- **`HistGradientBoostingRegressor`**, built into scikit-learn, histogram-based and fast. No extra
  dependency.
- **XGBoost**, the one that popularised the approach, with the most tuning knobs.
- **LightGBM**, which grows trees leaf-wise rather than level-wise and is usually the quickest on large
  feature sets.

> **CatBoost** is the fourth name you will meet, and it is strong on categorical features in particular.
> We do not install it for this course: it adds around 300 MB to the environment to demonstrate the same
> idea as the three above. If your data is mostly categorical, it is worth the download.

In [ ]:
boosting_models = {
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=0),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=4, random_state=0
    ),
    "LightGBM": lgb.LGBMRegressor(
        n_estimators=400, learning_rate=0.05, num_leaves=15, random_state=0, verbose=-1
    ),
}

for name, model in boosting_models.items():
    print(f"{name:<22} MAE = {evaluate(model, X_train, y_train, X_test, y_test):6.1f}")

All three land within 20 MAE of each other, which is the usual finding: the choice between boosting
libraries matters far less than the features you feed them. Pick one on practical grounds, speed or
dependency weight, rather than expecting an accuracy difference.

More surprising is that **all three lose to the random forest** on this series. That is a real result, not
a misconfiguration, and the next section takes it seriously.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-Tuning,-and-What-a-Validation-Score-Is-Worth">5. Tuning, and What a Validation Score Is Worth</h3>
</div>

The boosted models above use defaults chosen by hand. Boosting is the family that most rewards tuning, so
let us do it properly: a small grid, scored with `TimeSeriesSplit`, which builds folds that always train
on the past and test on the future.

In [ ]:
grid = list(itertools.product([0.03, 0.1], [7, 15, 31], [200, 600]))
results = []

for learning_rate, num_leaves, n_estimators in grid:
    candidate = lgb.LGBMRegressor(
        learning_rate=learning_rate, num_leaves=num_leaves,
        n_estimators=n_estimators, random_state=0, verbose=-1,
    )
    cv_score = -cross_val_score(
        candidate, X_train, y_train,
        cv=TimeSeriesSplit(5), scoring="neg_mean_absolute_error",
    ).mean()
    results.append({
        "learning_rate": learning_rate, "num_leaves": num_leaves,
        "n_estimators": n_estimators, "CV MAE": cv_score,
    })

tuning = pd.DataFrame(results).sort_values("CV MAE").reset_index(drop=True)
tuning.head(5).round({"CV MAE": 1})

In [ ]:
best = tuning.iloc[0]

tuned_lightgbm = lgb.LGBMRegressor(
    learning_rate=best["learning_rate"],
    num_leaves=int(best["num_leaves"]),
    n_estimators=int(best["n_estimators"]),
    random_state=0, verbose=-1,
)

tuned_score = evaluate(tuned_lightgbm, X_train, y_train, X_test, y_test)

print(f"Best by cross-validation: learning_rate={best['learning_rate']}, "
      f"num_leaves={int(best['num_leaves'])}, n_estimators={int(best['n_estimators'])}")
print(f"  Cross-validation MAE: {best['CV MAE']:.1f}")
print(f"  Held-out MAE:         {tuned_score:.1f}")
print(f"  Untuned LightGBM:     {evaluate(boosting_models['LightGBM'], X_train, y_train, X_test, y_test):.1f}")

Tuning helps: 268 down to 248. Note also which configuration won, the *smallest* of the models on offer,
with a low learning rate and only seven leaves. On 800 rows, the constraint that helps most is less
capacity.

Now look at the two MAE numbers for that same model. Cross-validation says 458, the held-out period says
248. The model did not improve by being deployed; the two numbers measure different things. Each
`TimeSeriesSplit` fold trains on a shorter history than the final model gets, and its first folds are
trained on a few hundred days, while the held-out period happens to be an easier stretch.

**Use cross-validation to rank candidates, not to predict what the error will be.** Notebook
[A06](./A06_Evaluating_models.ipynb) made this argument for forecast origins; it applies with just as much
force to model selection.

**Exercise.** Run the same grid with `KFold(5, shuffle=True)` instead of `TimeSeriesSplit`. Does it choose the same configuration? Compare the winner's held-out MAE against the one selected above.

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-The-Full-Comparison">6. The Full Comparison</h3>
</div>

Everything so far, plus the statistical comparators, on exactly the same 90-day period. The SARIMAX
specification is the one from Notebook [B02](./B02_ARIMA_models.ipynb), refitted here on this split so the
comparison is fair.

In [ ]:
exogenous = store[["Open", "Promo"]].astype(float).loc[X.index]

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    sarimax = ARIMA(
        y_train, exog=exogenous.iloc[:split],
        order=(1, 0, 1), seasonal_order=(1, 0, 1, 7),
    ).fit()
sarimax_forecast = sarimax.forecast(HOLDOUT_DAYS, exog=exogenous.iloc[split:])

# Baseline: the same weekday last week
seasonal_naive = y.shift(7).iloc[split:]

everything = {**linear_models, **tree_models, **boosting_models,
              "LightGBM (tuned)": tuned_lightgbm}

comparison = pd.DataFrame(
    [{"Model": name, "MAE": evaluate(model, X_train, y_train, X_test, y_test),
      "Family": "Machine learning"}
     for name, model in everything.items()]
    + [{"Model": "SARIMAX (B02)", "MAE": mean_absolute_error(y_test, sarimax_forecast),
        "Family": "Statistical"},
       {"Model": "Seasonal naive (weekly)", "MAE": mean_absolute_error(y_test, seasonal_naive),
        "Family": "Baseline"}]
).sort_values("MAE").reset_index(drop=True)

comparison.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

palette = {"Machine learning": "steelblue", "Statistical": "darkorange", "Baseline": "crimson"}
ordered = comparison.iloc[::-1]
ax.barh(ordered["Model"], ordered["MAE"], color=[palette[f] for f in ordered["Family"]])

for y_position, value in enumerate(ordered["MAE"]):
    ax.text(value + 15, y_position, f"{value:.0f}", va="center", fontsize=9)

handles = [plt.Rectangle((0, 0), 1, 1, color=colour) for colour in palette.values()]
ax.legend(handles, palette.keys(), loc="lower right")
ax.set_title("Held-out MAE over the same 90 days", fontsize=13, fontweight="bold")
ax.set_xlabel("MAE")
ax.set_xlim(0, ordered["MAE"].max() * 1.15)
ax.grid(axis="x", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

**The random forest wins**, at 226 against SARIMAX's 414 and the seasonal naive baseline's 1183. After
five notebooks in which simple methods kept embarrassing sophisticated ones, machine learning takes this
one clearly.

It is worth understanding why, because the reason is specific rather than general.

In [ ]:
# Why did the weekly baseline do so badly?
open_now = store["Open"].loc[y_test.index]
open_last_week = store["Open"].shift(7).loc[y_test.index]
pattern_broken = open_now != open_last_week

errors = (y_test - seasonal_naive).abs()

print(f"Days where the shop's opening differs from a week earlier: "
      f"{int(pattern_broken.sum())} of {len(y_test)}")
print(f"  MAE on those days:  {errors[pattern_broken].mean():7.0f}")
print(f"  MAE on the others:  {errors[~pattern_broken].mean():7.0f}")

There it is. Seven days in the test period had a different opening status from the same weekday a week
earlier, mostly public holidays, and on those seven days the baseline is off by 4,537 on average against
900 elsewhere. A handful of days accounts for most of its error.

The machine learning models beat it because they were **given the `open` column**, which is known months
in advance. This is not a triumph of non-linear modelling over linear, or of trees over ARIMA. It is the
value of an exogenous variable that tells you the answer on exactly the days the pattern breaks, and
SARIMAX improved for the same reason in Notebook B02.

The general lesson is the one this course keeps returning to from different directions: **information
beats algorithms**. The gap between the best and worst machine learning model here is smaller than the gap
opened by one well-chosen column.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-Where-Trees-Fail:-Extrapolation">7. Where Trees Fail: Extrapolation</h3>
</div>

Now the limitation, and it is fundamental rather than fixable.

A tree predicts by averaging the training values that land in a leaf. **Every prediction it can ever make
is an average of values it has already seen.** A tree-based model therefore cannot produce an output above
its training maximum or below its training minimum, no matter what the features say.

For a series with a trend, that is fatal. The following series is synthetic, so that the mechanism is
visible without argument: a steady upward trend, a seasonal cycle, and noise.

In [ ]:
rng = np.random.default_rng(0)
periods = 200
time_index = np.arange(periods)

trending = (
    100
    + 1.5 * time_index                                   # steady upward trend
    + 10 * np.sin(2 * np.pi * time_index / 12)           # seasonal cycle
    + rng.normal(0, 5, periods)                          # noise
)

trend_features = pd.DataFrame({
    "t": time_index,
    "sin": np.sin(2 * np.pi * time_index / 12),
    "cos": np.cos(2 * np.pi * time_index / 12),
})

trend_split = 150
predictions = {}

for name, model in [("Linear regression", LinearRegression()),
                    ("Random forest", RandomForestRegressor(n_estimators=200, random_state=0))]:
    model.fit(trend_features.iloc[:trend_split], trending[:trend_split])
    predictions[name] = model.predict(trend_features.iloc[trend_split:])
    print(f"{name:<20} MAE = {mean_absolute_error(trending[trend_split:], predictions[name]):6.1f}")

print()
print(f"Training range: {trending[:trend_split].min():.0f} to {trending[:trend_split].max():.0f}")
print(f"Test range:     {trending[trend_split:].min():.0f} to {trending[trend_split:].max():.0f}")
print()
print("Final point:")
print(f"  actual                     {trending[-1]:6.0f}")
for name, prediction in predictions.items():
    print(f"  {name:<25}{prediction[-1]:6.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))

ax.plot(time_index[:trend_split], trending[:trend_split], color="steelblue",
        linewidth=1.2, label="Train")
ax.plot(time_index[trend_split:], trending[trend_split:], color="black",
        linewidth=1.8, label="Actual")
ax.plot(time_index[trend_split:], predictions["Linear regression"], color="seagreen",
        linewidth=1.6, linestyle="--", label="Linear regression")
ax.plot(time_index[trend_split:], predictions["Random forest"], color="crimson",
        linewidth=1.6, linestyle="--", label="Random forest")

ax.axhline(trending[:trend_split].max(), color="gray", linestyle=":", linewidth=1.2)
ax.text(5, trending[:trend_split].max() + 4, "highest value seen in training",
        fontsize=9, color="gray")
ax.axvline(trend_split, color="gray", linestyle="--", linewidth=1.0)

ax.set_title("A tree model cannot predict above what it has seen",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Time")
ax.set_ylabel("Value")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The random forest flattens out against the dotted line, which is the largest value in its training data.
Its final prediction is 324 where the truth is 396, and no amount of tuning, extra trees or better
features will change that: the number 396 is not available to it.

Linear regression has no such limit and tracks the trend almost exactly, with an MAE of 4 against the
forest's 37.

**This is why the Rossmann results above are not a general claim about machine learning.** That series has
no meaningful trend, so nothing was being asked of the models that they could not do. Put the same forest
on a growing series and it will fail in this specific, structural way.

Three ways out, all of which amount to removing the trend before the model sees it:

1. **Difference the target.** Predict the change rather than the level, then add it back. The changes are
   stationary even when the level is not.
2. **Detrend explicitly.** Fit a trend, model the residuals with the tree, add the trend back for the
   forecast. This is what Notebook B03's dynamic harmonic regression does with its deterministic terms.
3. **Use a model that can extrapolate.** Linear models, and the neural networks of Part D, have no
   equivalent ceiling.

That third option is a good reason to keep a linear model in the comparison even when it loses.

**Exercise.** Apply fix 1 to the synthetic series: train the random forest on `np.diff(trending)` instead of the level, then cumulatively sum its predictions back onto the last training value. How close does it get to the linear model?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="8.-Choosing-a-Model">8. Choosing a Model</h3>
</div>

| Model | Reach for it when | Watch out for |
|---|---|---|
| **Linear / Ridge / Lasso** | You need interpretability, or the series trends | Misses interactions and non-linearity |
| **Decision tree** | You want to see the rules | Overfits badly on its own |
| **Random forest** | A strong result with almost no tuning | Cannot extrapolate; larger models |
| **Gradient boosting** | Top accuracy, many features, time to tune | Cannot extrapolate; will fit noise if over-configured |

What this notebook actually established, in order of how much it should change your practice:

**Information beats algorithms.** One known-in-advance column, `open`, explains more of the gap between
the best and worst forecast here than every modelling choice put together.

**Trees cannot extrapolate.** Check whether your series trends before reaching for one. If it does,
difference or detrend first, and keep a linear model in the comparison.

**The boosting libraries are interchangeable.** Within 20 MAE of each other. Choose on speed and
dependencies, and spend the saved time on features.

**Untuned is often enough.** The random forest, with no tuning at all, beat every tuned boosted model on
this series. Tune when you have the validation budget to do it honestly, and do not assume it will
overturn a well-chosen default.

**A validation score ranks models; it does not predict your error.** 458 in cross-validation, 248 on the
held-out period, same model.

---

Every model in this notebook was judged on its own. The next one stops choosing between them and combines
them instead, which turns out to be more reliable than picking a winner:
[C03 - Ensembles and Model Combinations](./C03_Ensembles.ipynb).

**Solutions.** Worked answers to the 2 exercises above, with the reasoning behind them, are in
[C02_Machine_learning_models_solutions.ipynb](../solutions/C02_Machine_learning_models_solutions.ipynb). Try each one yourself first.
